# 練習問題

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kyorin-phys/MLIntro2026/blob/main/4/ex4.ipynb)

## MedMNIST
[MedMNIST](https://medmnist.com/) は、公開されている医療画像を MNIST と同じ28×28ピクセルに縮小して集めたデータセットである。`pip install medmnist` でインストールすると、Pythonから数行でダウンロードして使える。

ここでは PneumoniaMNIST（小児の胸部X線画像、正常=0 / 肺炎=1）を使って、mnist.ipynb と同じ方法で分類する。

28×28 は臨床で使う画像よりはるかに粗いので、あくまで練習用のデータである。

In [ ]:
!pip install -q medmnist
import numpy as np
import matplotlib.pyplot as plt
import medmnist
from medmnist import PneumoniaMNIST

info = medmnist.INFO['pneumoniamnist'] # データセットの説明
print(info['description'])
print(info['label'])   # ラベルの番号と意味
print(info['license']) # 利用条件

## 練習問題1
訓練データとテストデータを読み込み、画像の枚数と形、ラベル0（正常）と1（肺炎）の枚数を表示せよ。また、肺炎の割合を訓練データとテストデータで比べよ。最後に、最初の10枚の画像をラベル付きで表示せよ。

In [ ]:
train = PneumoniaMNIST(split='train', download=True)
test = PneumoniaMNIST(split='test', download=True)

# 画素値 0〜255 を 0〜1 に正規化する
train_images = train.imgs / 255.0
test_images = test.imgs / 255.0
# ラベルは (枚数, 1) の形なので、flatten() で1次元にする
train_labels = train.labels.flatten()
test_labels = test.labels.flatten()

# 以下にコードを記述

In [ ]:
#@title 解答例（「コードを表示」で見られる）
print(train_images.shape, test_images.shape)
print(np.bincount(train_labels), np.bincount(test_labels)) # ラベル0, 1 それぞれの枚数
print('肺炎の割合 訓練:', train_labels.mean(), ' テスト:', test_labels.mean()) # ラベルが0/1なので平均が1の割合になる

In [ ]:
#@title 解答例（最初の10枚をラベル付きで表示）
plt.figure(figsize=(15, 5))
for i in range(10):
    plt.subplot(2, 5, i + 1)
    plt.imshow(train_images[i], cmap='gray')
    plt.title(f'label: {train_labels[i]}')
    plt.axis('off')
plt.show()

### 訓練データの比率をそろえる
練習問題1で見たように、訓練データは約3/4が肺炎で、正常が少ない。このまま学習すると、モデルは多い方の「肺炎」と答えがちになり、感度は高いが特異度は低くなりやすい。正解率だけを見ているとこの偏りに気づきにくい。

そこで、**訓練データだけ**、各クラスから一番少ないクラスと同じ枚数を無作為に取り出して比率をそろえる（アンダーサンプリング）。捨てる画像が出る代わりに、学習が偏りにくくなり、学習時間も短くなる。テストデータは実際の状況を表すものなのでそのままにする。

下のセルを実行する（練習問題4でも同じ関数を使う）。

In [ ]:
def balance(images, labels, seed=0):
    """各クラスから、一番少ないクラスと同じ枚数だけ無作為に取り出す"""
    rng = np.random.default_rng(seed)
    n = np.bincount(labels).min() # 一番少ないクラスの枚数
    idx = np.concatenate([rng.choice(np.where(labels == k)[0], n, replace=False) # クラスkの中からn枚を選ぶ
                          for k in np.unique(labels)])
    rng.shuffle(idx) # クラスの順に並ばないように混ぜる
    return images[idx], labels[idx]

print('そろえる前:', np.bincount(train_labels))
train_images, train_labels = balance(train_images, train_labels)
print('そろえた後:', np.bincount(train_labels))

## 練習問題2
mnist.ipynb と同じモデルで学習させ、学習曲線（損失と正解率の推移）を表示し、テストデータで正解率を求めよ。

分類するクラスが10種類（数字0〜9）から2種類（正常、肺炎）になるので、出力層を `Dense(2, activation='softmax')` に変える。それ以外はMNISTと同じでよい。

テストデータの肺炎の割合を練習問題1の枚数から計算し、得られた正解率と比べてみよ（すべて「肺炎」と答えた場合の正解率はいくらか）。

In [ ]:
from tensorflow.keras.layers import Flatten, Dense, Input
from tensorflow.keras.models import Sequential
from tensorflow.keras.losses import SparseCategoricalCrossentropy
from tensorflow.keras.utils import set_random_seed

set_random_seed(0) # 乱数を固定して毎回同じ結果にする

# 以下にコードを記述

In [ ]:
#@title 解答例（「コードを表示」で見られる）
model = Sequential([
    Input(shape=(28,28)), # 入力
    Flatten(), # 画像をベクトルにする
    Dense(128, activation='relu'), # 中間
    Dense(2, activation='softmax') # 出力: 正常(0)と肺炎(1)の確率
])
model.compile(optimizer='adam', loss=SparseCategoricalCrossentropy(), metrics=['accuracy'])
history = model.fit(train_images, train_labels, epochs=10, validation_data=(test_images, test_labels))
test_loss, test_acc = model.evaluate(test_images, test_labels)
print(f'Test accuracy: {test_acc}')

# 学習曲線をプロット（左: 損失、右: 正解率）
import pandas as pd
pd.DataFrame(history.history).plot(subplots=[('loss', 'val_loss'), ('accuracy', 'val_accuracy')], layout=(1, 2), figsize=(10, 4), xlabel='Epoch')
plt.show()

## 練習問題3
肺炎（1）を陽性として、テストデータの混同行列、感度、特異度を求めよ。また、ROC曲線を描いてAUCを求めよ（第3回の乳癌の例を参考にする）。

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, RocCurveDisplay

# 以下にコードを記述

In [ ]:
#@title 解答例（「コードを表示」で見られる）
prob = model.predict(test_images)[:, 1] # 肺炎である確率（出力の2列目）
y_pred = (prob >= 0.5).astype(int)      # 確率0.5以上なら肺炎と予測
tp, fn, fp, tn = confusion_matrix(test_labels, y_pred, labels=[1, 0]).ravel() # 陽性(1)を先にする（第3回と同じ並び）
print('感度:', tp / (tp + fn))
print('特異度:', tn / (tn + fp))
ConfusionMatrixDisplay.from_predictions(test_labels, y_pred, labels=[1, 0], display_labels=['pneumonia', 'normal'])
plt.show()
RocCurveDisplay.from_predictions(test_labels, prob, name='pneumonia')
plt.show()

## 練習問題4（多クラス分類）
練習問題1〜3は2クラス（正常 / 肺炎）だった。ここでは OCTMNIST（網膜の光干渉断層像 OCT、4種類）を使って、**3クラス以上**の分類を同じ流れで行う。

ラベル：0 = CNV（脈絡膜新生血管）、1 = DME（糖尿病黄斑浮腫）、2 = drusen（ドルーゼン）、3 = normal（正常）

### 2クラスと多クラスで変わるところ

| | 2クラス（PneumoniaMNIST） | 多クラス（OCTMNIST, K=4） |
|---|---|---|
| 出力層 | `Dense(2, activation='softmax')` | `Dense(K, activation='softmax')` |
| 損失関数 | `SparseCategoricalCrossentropy()` | 同じ（ラベルは 0〜K-1 の整数のまま） |
| 予測クラス | 陽性の確率が0.5以上なら1 | 確率が最大のクラス `np.argmax(prob, axis=1)` |
| 混同行列 | 2×2（`labels=[1, 0]` で [[TP, FN], [FP, TN]]） | K×K（行=正解、列=予測） |
| 感度・特異度 | 陽性クラスについて1つ | **クラスごと**に「そのクラス 対 それ以外」で求める |
| ROC / AUC | 1本 | クラスごとに1本。平均は `roc_auc_score(..., multi_class='ovr')` |
| 比較の基準 | 全部「テストデータで多い方のクラス」と答えたときの正解率 | 同じ（テストデータは各クラス同数なので 0.25） |

モデルの部分はほとんど変わらず、変わるのは**評価の仕方**である。


In [ ]:
from medmnist import OCTMNIST

info = medmnist.INFO['octmnist']
print(info['label'])
K = len(info['label']) # クラス数
class_names = ['CNV', 'DME', 'drusen', 'normal']

train4 = OCTMNIST(split='train', download=True)
test4 = OCTMNIST(split='test', download=True)
train4_images = train4.imgs / 255.0
test4_images = test4.imgs / 255.0
train4_labels = train4.labels.flatten()
test4_labels = test4.labels.flatten()

# 以下にコードを記述（画像の形と、クラスごとの枚数を訓練・テストで比べる）

In [ ]:
#@title 解答例（「コードを表示」で見られる）
print(train4_images.shape, test4_images.shape)
print('訓練:', np.bincount(train4_labels))
print('テスト:', np.bincount(test4_labels))

### 訓練データの比率をそろえる
訓練データはクラスごとの枚数に大きな差があり、drusen が一番少ない（テストデータは各クラス同じ枚数）。前半と同じく `balance` で訓練データの比率をそろえる。

In [ ]:
train4_images, train4_labels = balance(train4_images, train4_labels)
print('そろえた後:', np.bincount(train4_labels))

### 学習
練習問題2と同じモデルで、出力層だけ `Dense(K, activation='softmax')` に変える。


In [ ]:
import pandas as pd
set_random_seed(0)

# 以下にコードを記述

In [ ]:
#@title 解答例（「コードを表示」で見られる）
model4 = Sequential([
    Input(shape=(28,28)),
    Flatten(),
    Dense(128, activation='relu'),
    Dense(K, activation='softmax') # 出力: K個のクラスの確率
])
model4.compile(optimizer='adam', loss=SparseCategoricalCrossentropy(), metrics=['accuracy'])
history4 = model4.fit(train4_images, train4_labels, epochs=10, validation_data=(test4_images, test4_labels))
test_loss, test_acc = model4.evaluate(test4_images, test4_labels)
print(f'Test accuracy: {test_acc}')
pd.DataFrame(history4.history).plot(subplots=[('loss', 'val_loss'), ('accuracy', 'val_accuracy')], layout=(1, 2), figsize=(10, 4), xlabel='Epoch')
plt.show()

### 評価
正解率だけでなく、混同行列とクラスごとの感度・特異度を求める。正解率が同じでも、よく当たるクラスとほとんど当たらないクラスが混ざっていることがある。


In [ ]:
from sklearn.metrics import classification_report, roc_auc_score

# 以下にコードを記述

In [ ]:
#@title 解答例（「コードを表示」で見られる）
prob4 = model4.predict(test4_images)  # 形は (枚数, K)。各行の合計は1
y_pred4 = np.argmax(prob4, axis=1)    # 確率が最大のクラスを予測とする

# 混同行列 (K×K)
cm4 = confusion_matrix(test4_labels, y_pred4)
ConfusionMatrixDisplay(cm4, display_labels=class_names).plot()
plt.show()

# クラスごとの感度・特異度（そのクラスを陽性、それ以外を陰性とみなす）
for k in range(K):
    tp = cm4[k, k]
    fn = cm4[k, :].sum() - tp  # 正解がkなのに、k以外と予測
    fp = cm4[:, k].sum() - tp  # 正解がk以外なのに、kと予測
    tn = cm4.sum() - tp - fn - fp
    print(f'{class_names[k]:7s} 感度: {tp/(tp+fn):.3f}  特異度: {tn/(tn+fp):.3f}')

# precision, recall(=感度), f1 をまとめて表示
print(classification_report(test4_labels, y_pred4, target_names=class_names))

# AUC（クラスごとに「そのクラス 対 それ以外」で求めて平均）
print('AUC (one-vs-rest):', roc_auc_score(test4_labels, prob4, multi_class='ovr'))
for k in range(K):
    RocCurveDisplay.from_predictions(test4_labels == k, prob4[:, k], name=class_names[k], ax=plt.gca())
plt.show()

### 結果の読み方
正解率だけを見ると全体としてそこそこ当たっているように見えるが、混同行列を見ると drusen はほとんど当たらず、多くが CNV や normal と予測されている。このように、**正解率が同じでもクラスごとの性能は大きく違う**ことがある。

drusen（ドルーゼン）は網膜の下にたまる小さな沈着物で、加齢黄斑変性の初期に見られる。CNV（加齢黄斑変性が進んだ状態）の眼にもドルーゼンがあることが多く、また沈着物が小さければ正常に近い像になる。28×28 に縮小するとこうした細かい違いは消えてしまうので、CNV とも normal とも区別しにくい。

訓練データの比率をそろえても、少ないクラスが無視されにくくなるだけで、画像から**見分ける力**そのものが上がるわけではない（比率をそろえるために訓練データを2/3近く捨てていることにも注意）。改善するには、画像の位置関係を利用する畳み込みニューラルネットワーク（CNN）を使う、より大きな画像（`OCTMNIST(split='train', download=True, size=64)` など）を使う、といった方法がある。

### 他のデータでも試す*
データの一覧は `medmnist.INFO.keys()` で確認できる。`medmnist.INFO['データ名']['task']` が `multi-class` のものは上と同じ方法でできる。

例：BloodMNIST（血液細胞の顕微鏡画像、8種類、カラー）
* `from medmnist import BloodMNIST` として、`OCTMNIST` を `BloodMNIST` に置き換える
* カラー画像なので画像の形は (28, 28, 3) になる。入力層を `Input(shape=(28,28,3))` にする
* 8種類の分類なので K=8 になる

注意：ChestMNIST は `multi-label`（1枚の画像に複数の所見がありうる）なので、この方法ではできない。

利用条件はデータごとに異なるので `medmnist.INFO['データ名']['license']` で確認すること。


In [ ]:
print(list(medmnist.INFO.keys()))